<a href="https://colab.research.google.com/github/Atharv-1905/Machine-Learning/blob/practice/Predicting_Student_Scores(With_Hyperparameter_Tuning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from xgboost import XGBRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

In [2]:
SEED = 42

In [3]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [4]:
ID_COL = 'id'
TARGET_COL = 'exam_score'

In [5]:
train_df_cleaned = train_df.dropna(subset=[TARGET_COL])
X = train_df_cleaned.drop(columns=[ID_COL, TARGET_COL], errors='ignore')
y = train_df_cleaned[TARGET_COL]

In [6]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Detected: {len(num_cols)} Numeric columns, {len(cat_cols)} Categorical columns")

Detected: 4 Numeric columns, 7 Categorical columns


In [7]:
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])



cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])



preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])



full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(objective='reg:squarederror', random_state=SEED, tree_method='hist', device='cuda'))
])

In [8]:
param_dist = {
    'model__n_estimators': [100, 300, 500, 1000],   # Number of trees
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2], # How fast it learns
    'model__max_depth': [3, 4, 5, 6],               # How complex the trees are
    'model__subsample': [0.7, 0.8, 0.9],            # Fraction of data to use
    'model__colsample_bytree': [0.7, 0.8, 0.9]      # Fraction of features to use
}

search = RandomizedSearchCV(
    full_pipeline,
    param_distributions=param_dist,
    n_iter=20,          # Try 20 random combinations
    cv=3,               # 3-Fold Cross Validation (Reliability check)
    scoring='neg_root_mean_squared_error',
    n_jobs=1,
    verbose=1,
    random_state=SEED
)

In [9]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [10]:
search.fit(X_train, y_train)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:774: UserWarning: [13:49:07] WARNING: /workspace/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['age',
                                                                                'study_hours',
                                                                                'class_attendance',
                                                                                'sleep_hours']),
                                                                              ('cat',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(fill_value='Missing',
                                                                                                              strategy='constant')),
                                                                                               ('encod...
                                                           n_estimators=None,
                                                           n_jobs=None,
                                                           num_parallel_tree=None, ...))]),
                   n_iter=20, n_jobs=1,
                   param_distributions={'model__colsample_bytree': [0.7, 0.8,
                                                                    0.9],
                                        'model__learning_rate': [0.01, 0.05,
                                                                 0.1, 0.2],
                                        'model__max_depth': [3, 4, 5, 6],
                                        'model__n_estimators': [100, 300, 500,
                                                                1000],
                                        'model__subsample': [0.7, 0.8, 0.9]},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=1)

In [11]:
best_model = search.best_estimator_

print(f"\nBest Params: {search.best_params_}")
print(f"Best CV RMSE: {-search.best_score_:.4f}")


Best Params: {'model__subsample': 0.8, 'model__n_estimators': 500, 'model__max_depth': 4, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.8}
Best CV RMSE: 8.8498


In [12]:
y_pred_valid = best_model.predict(X_valid)
rmse = root_mean_squared_error(y_valid, y_pred_valid)
r2 = r2_score(y_valid, y_pred_valid)

print("\n--- 🏁 Final Performance Report ---")
print(f"RMSE: {rmse:.4f}")
print(f"R2 Score: {r2:.4f}")


--- 🏁 Final Performance Report ---
RMSE: 8.7401
R2 Score: 0.7872


In [13]:
submission_ids = test_df[ID_COL]

X_test_final = test_df.drop(columns=[ID_COL], errors='ignore')

final_predictions = best_model.predict(X_test_final)

submission = pd.DataFrame({
    'id': submission_ids,
    'exam_score': final_predictions
})

submission.to_csv('submission_optimized.csv', index=False)